# E3 — Entrenamiento PoinTr v3 (alineación original, 200 épocas)

**Modelo:** PoinTr (Yu et al., 2021) — transformer para shape completion.

**Diferencias respecto a v2:**
- **`CENTRAR_EN_ROTO=False`** — la nube rota NO se centra en su propio centroide; roto y completo comparten el mismo frame de coordenadas original. Los puntos del fragmento caen exactamente sobre la región correspondiente del completo.
- **200 épocas** (v2 usó 150 y el modelo seguía mejorando)
- **Entrenamiento desde cero** — sin resume de checkpoints de v2 (CKPT_DIR nuevo)
- Misma blacklist y mismos datos limpios que v2

**Contexto sobre `CENTRAR_EN_ROTO`:** en v5 (PCN) se activó el centrado en el fragmento (H19) para evitar que el modelo aprendiera la traslación. Sin embargo, esto desacopla roto del completo — la parte existente ya no está superpuesta sobre la región correspondiente del objeto completo. Con `False`, la geometría es más intuitiva: el modelo ve el fragmento en su posición original y predice el objeto entero en ese mismo frame.

**Métricas de referencia:**
- PCN v5: CD=0.0630, F-Score=0.0257
- PoinTr v2 (datos limpios, 150 ep, checkpoint contaminado): CD=0.0533, F-Score=0.3278

⏱️ **Tiempo estimado:**
- A100: ~2-2.5 horas (200 épocas, batch=32, ~321 pares train)
- T4: ~5-6 horas

⚠️ **GPU obligatoria.** Menú → Entorno de ejecución → Cambiar tipo → T4 o A100 GPU

In [ ]:
# ── CELDA 1: Montar Drive ──────────────────────────────────────
from google.colab import drive
import os

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')
    print('Drive montado.')

In [ ]:
# ── CELDA 2: Clonar repos + instalar + compilar CUDA ──────────
#
# Esta celda tarda ~10-15 minutos la primera vez (compilación CUDA).
# Si se interrumpe y se relanza, la compilación salta porque pip detecta
# que el paquete ya está instalado.

import os
import subprocess
import time
from getpass import getpass

# ── 1. Clonar PoinTr oficial ──────────────────────────────────
if not os.path.exists('/content/PoinTr'):
    print('Clonando PoinTr...')
    r = subprocess.run(
        ['git', 'clone', 'https://github.com/yuxumin/PoinTr',
         '/content/PoinTr', '--depth=1', '-q'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('ERROR al clonar PoinTr:', r.stderr)
    else:
        print('PoinTr clonado.')
else:
    print('[OK] /content/PoinTr ya existe.')

# ── 2. Clonar repo TFM ───────────────────────────────────────
REPO_DIR = '/content/TFM'
if not os.path.exists(REPO_DIR):
    token = getpass('Token GitHub (ghp_...): ')
    repo_url = f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D'
    subprocess.run(['git', 'clone', repo_url, REPO_DIR, '-q'],
                   capture_output=True, text=True)
    del token
    print('Repo TFM clonado.')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'],
                   capture_output=True)
    print('[OK] /content/TFM ya existe — actualizado.')

os.chdir(REPO_DIR)
subprocess.run(['git', 'checkout', 'raquel/e3', '-q'], capture_output=True)
print(f'Directorio: {os.getcwd()}')

# ── 3. Instalar dependencias Python ──────────────────────────
print('\nInstalando dependencias Python...')
subprocess.run(['pip', 'install', 'timm', 'easydict', 'pyyaml', '--quiet'])
print('[OK] timm, easydict, pyyaml')

# ── 4. Compilar extensiones CUDA de PoinTr ───────────────────
# pointnet2_ops: KNN + ball query en CUDA (necesario para el encoder DGCNN)
# chamfer_dist:  Chamfer Distance en CUDA (~3x más rápido que cdist)

CUDA_OK = True

for ext_name, ext_path in [
    ('pointnet2_ops', '/content/PoinTr/extensions/pointnet2_ops_lib'),
    ('chamfer_dist',  '/content/PoinTr/extensions/chamfer_dist'),
]:
    print(f'\nCompilando {ext_name}...', flush=True)
    t0 = time.time()
    r = subprocess.run(
        ['pip', 'install', '-e', ext_path, '--quiet'],
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    if r.returncode == 0:
        print(f'  [OK] {ext_name} compilado en {elapsed:.0f}s')
    else:
        print(f'  [WARN] {ext_name} falló ({elapsed:.0f}s) — se usará fallback PyTorch')
        print(f'  Error: {r.stderr[-500:]}')
        CUDA_OK = False

print(f'\nResumen: CUDA extensions = {"OK" if CUDA_OK else "fallback PyTorch"}')
print('Celda 2 completa.')

In [ ]:
# ── CELDA 3: RUTAS DE DRIVE ────────────────────────────────────
DRIVE   = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'

VERSION = 'v3_pointr'

RUTA_SINTETICO    = f'{DRIVE}/Datos_E2_E3/General/sintetico_roturas_centradas'
RUTA_FB_PROCESADO = f'{DRIVE}/Datos_E2_E3/General/Fantastik_Break_Preprocesado'
RUTA_BLACKLIST    = f'{DRIVE}/Datos_E2_E3/General/blacklist_sintetico_roturas_centradas.txt'

RUTA_SALIDA_MODELO     = f'{BASE_E3}/modelos/{VERSION}'
RUTA_SALIDA_RESULTADOS = f'{BASE_E3}/resultados/{VERSION}'

print(f'Rutas PoinTr {VERSION}:')
print(f'  sintetico       : {RUTA_SINTETICO}')
print(f'  fantastic_breaks: {RUTA_FB_PROCESADO}')
print(f'  blacklist       : {RUTA_BLACKLIST}')
print(f'  salida modelo   : {RUTA_SALIDA_MODELO}')
print(f'  salida resultados: {RUTA_SALIDA_RESULTADOS}')

In [ ]:
# ── CELDA 3b: Cargar blacklist desde Drive ─────────────────────
from pathlib import Path

blacklist_set = set()
bl_path = Path(RUTA_BLACKLIST)
if bl_path.exists():
    with open(bl_path) as f:
        for line in f:
            stem = line.strip()
            if stem:
                blacklist_set.add(stem)
    print(f'Blacklist cargada: {len(blacklist_set)} pares a excluir')
else:
    print(f'[WARN] Blacklist no encontrada en:')
    print(f'  {RUTA_BLACKLIST}')
    print('  Comprueba la ruta. Sin blacklist se usarían todos los pares (datos sucios).')
    raise FileNotFoundError('Blacklist no encontrada — ajusta RUTA_BLACKLIST en Celda 3')

In [ ]:
# ── CELDA 4: Verificar rutas ────────────────────────────────────
from pathlib import Path

rutas = {
    'sintetico_roturas_centradas' : RUTA_SINTETICO,
    'fantastic_breaks'            : RUTA_FB_PROCESADO,
    'blacklist'                   : RUTA_BLACKLIST,
}

for nombre, ruta in rutas.items():
    existe = Path(ruta).exists()
    estado = 'OK' if existe else 'FALTA'
    print(f'  [{estado}] {nombre}: {ruta}')

n_sint = len(list(Path(RUTA_SINTETICO).glob('*_completo.npy'))) if Path(RUTA_SINTETICO).exists() else 0
n_fb   = len(list(Path(RUTA_FB_PROCESADO).glob('*_completo.npy'))) if Path(RUTA_FB_PROCESADO).exists() else 0
print(f'\nPares disponibles (antes de filtrar):')
print(f'  sintetico       : {n_sint}')
print(f'  fantastic_breaks: {n_fb}')
print(f'  TOTAL           : {n_sint + n_fb}')

In [ ]:
# ── CELDA 5: Copiar datos desde Drive ──────────────────────────
import subprocess
from pathlib import Path

def copiar_dir(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists() and any(dst.glob('*.npy')):
        n = len(list(dst.glob('*.npy')))
        print(f'  [OK] ya existe: {dst.name}  ({n} .npy)')
        return
    if not src.exists():
        print(f'  [ERROR] no encontrado: {src}')
        return
    dst.mkdir(parents=True, exist_ok=True)
    print(f'  Copiando {src.name}...', flush=True)
    r = subprocess.run(['rsync', '-a', '--no-links', f'{src}/', str(dst)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  [ERROR rsync] {r.stderr[:300]}')
    else:
        n = len(list(dst.glob('*.npy')))
        print(f'  listo — {n} archivos .npy copiados.')

copiar_dir(RUTA_SINTETICO,    'Datos/sintetico/roturas_centradas')
copiar_dir(RUTA_FB_PROCESADO, 'Datos/fantastic_breaks/procesado')

print()
for c in ['Datos/sintetico/roturas_centradas', 'Datos/fantastic_breaks/procesado']:
    n = len(list(Path(c).glob('*.npy'))) if Path(c).exists() else 0
    print(f'  {c}: {n} .npy')

In [ ]:
# ── CELDA 6: Verificar GPU ─────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'PyTorch CUDA: {torch.version.cuda}')
else:
    print('Sin GPU. Ve a Entorno de ejecución → Cambiar tipo → A100 o T4')

In [ ]:
# ── CELDA 7: ENTRENAR PoinTr v3 (CENTRAR_EN_ROTO=False, 200 épocas) ────

import sys, os, math, time, types, glob as _glob, shutil, random
from pathlib import Path

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from easydict import EasyDict

# ── Dispositivo ──────────────────────────────────────────────
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_STR = str(device)
print(f'Dispositivo: {device}')
if device.type == 'cpu':
    print('AVISO: sin GPU. Cambia a T4/L4/A100 y reconecta.')

# ══════════════════════════════════════════════════════════════
# 0. LIMPIAR CACHÉ DE MÓDULOS DE PoinTr
# ══════════════════════════════════════════════════════════════
_pointr_prefixes = ('models', 'utils.registry', 'utils.config', 'utils.logger',
                    'utils.misc', 'extensions', 'datasets', 'tools')
_to_del = [k for k, v in sys.modules.items()
           if (any(k == p or k.startswith(p + '.') for p in _pointr_prefixes)
               or (hasattr(v, '__file__') and v.__file__
                   and '/content/PoinTr' in str(v.__file__)))]
for _k in _to_del: del sys.modules[_k]
print(f'[reset] {len(_to_del)} módulos PoinTr eliminados del caché')

# ══════════════════════════════════════════════════════════════
# 1. PATCH SOURCE DE PoinTr — .cuda() → .to(device)
# ══════════════════════════════════════════════════════════════
_n_patched = 0
for _fp in (_glob.glob('/content/PoinTr/models/*.py') +
            _glob.glob('/content/PoinTr/models/**/*.py')):
    try:
        with open(_fp, encoding='utf-8') as _f: _src = _f.read()
        _new = _src.replace('.cuda()', f'.to("{DEVICE_STR}")')
        if _new != _src:
            with open(_fp, 'w', encoding='utf-8') as _f: _f.write(_new)
            _n_patched += 1
    except Exception: pass
print(f'[patch] {_n_patched} archivos PoinTr: .cuda() → .to("{DEVICE_STR}")')

# ══════════════════════════════════════════════════════════════
# 2. MOCKS DE EXTENSIONES CUDA
# ══════════════════════════════════════════════════════════════
def _dummy_cls(name):
    return type(name, (nn.Module,), {
        '__init__': lambda self, *a, **kw: super(type(self), self).__init__(),
        'forward':  lambda self, x, *a, **kw: x,
    })

def _force(mod_name, attrs):
    m = types.ModuleType(mod_name)
    for k, v in attrs.items(): setattr(m, k, v)
    sys.modules[mod_name] = m

def _inject(mod_name, attrs):
    if mod_name not in sys.modules: _force(mod_name, attrs)

def _chamfer_raw(a, b):
    dist = torch.cdist(a, b, p=2)
    return dist.min(dim=2).values, dist.min(dim=1).values

class _ChamferL1(nn.Module):
    def forward(self, a, b):
        d1, d2 = _chamfer_raw(a.contiguous(), b.contiguous())
        return (d1.mean() + d2.mean()) / 2

class _ChamferL2(nn.Module):
    def forward(self, a, b):
        d1, d2 = _chamfer_raw(a.contiguous(), b.contiguous())
        return ((d1**2).mean() + (d2**2).mean()) / 2

class _ChamferL1_PM(nn.Module):
    def forward(self, a, b):
        return torch.cdist(a.contiguous(), b.contiguous(), p=2).min(dim=2).values.mean()

_ch_attrs = {'ChamferDistanceL1': _ChamferL1, 'ChamferDistanceL2': _ChamferL2,
             'ChamferDistanceL1_PM': _ChamferL1_PM, 'chamfer_3DDist': _chamfer_raw}
for _n in ['chamfer', 'chamfer_dist', 'extensions.chamfer_dist',
           'chamfer3D', 'chamfer3D.dist_chamfer_3D']:
    _force(_n, _ch_attrs)
print('[OK] Mock chamfer')

if 'pointnet2_ops' not in sys.modules:
    def _fps(xyz, npoint):
        B, N, _ = xyz.shape; dev = xyz.device
        idx = torch.zeros(B, npoint, dtype=torch.int32, device=dev)
        dist = torch.full((B, N), 1e10, device=dev)
        farthest = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
        bi = torch.arange(B, dtype=torch.long, device=dev)
        for i in range(npoint):
            idx[:, i] = farthest.int()
            c = xyz[bi, farthest].unsqueeze(1)
            dist = torch.min(dist, ((xyz - c)**2).sum(-1))
            farthest = dist.max(-1)[1]
        return idx
    def _gather_op(features, idx):
        B, C, N = features.shape; M = idx.shape[1]
        return features.gather(2, idx.long().unsqueeze(1).expand(B, C, M)).contiguous()
    def _ball_query(radius, nsample, xyz, new_xyz):
        dists = torch.cdist(new_xyz.float(), xyz.float())
        srt = dists.argsort(dim=-1); topk = srt[:, :, :nsample]
        return torch.where(dists.gather(2, topk) > radius,
                           srt[:, :, :1].expand_as(topk), topk).int()
    def _grouping_op(features, idx):
        B, C, N = features.shape; S, K = idx.shape[1], idx.shape[2]
        flat = idx.long().view(B, 1, S*K).expand(B, C, S*K)
        return features.gather(2, flat).view(B, C, S, K).contiguous()
    def _three_nn(unknown, known):
        dists = torch.cdist(unknown.float(), known.float())
        dist2, idx = dists.topk(3, dim=-1, largest=False)
        return dist2.float(), idx.int()
    def _three_interp(features, idx, weight):
        B, C, M = features.shape; N = idx.shape[1]
        flat = idx.long().view(B, 1, N*3).expand(B, C, N*3)
        return (features.gather(2, flat).view(B, C, N, 3) * weight.unsqueeze(1)).sum(-1).contiguous()
    _utils = types.ModuleType('pointnet2_ops.pointnet2_utils')
    for k, v in {'furthest_point_sample': _fps, 'gather_operation': _gather_op,
                 'ball_query': _ball_query, 'grouping_operation': _grouping_op,
                 'three_nn': _three_nn, 'three_interpolate': _three_interp}.items():
        setattr(_utils, k, v)
    _pm2 = types.ModuleType('pointnet2_ops'); _pm2.pointnet2_utils = _utils
    sys.modules['pointnet2_ops'] = _pm2; sys.modules['pointnet2_ops.pointnet2_utils'] = _utils
    print('[OK] Mock pointnet2_ops')
else:
    print('[OK] pointnet2_ops ya disponible')

class _KNN(nn.Module):
    def __init__(self, k, transpose_mode=False):
        super().__init__(); self.k = k; self.transpose_mode = transpose_mode
    def forward(self, ref, query):
        if self.transpose_mode:
            dists = torch.cdist(query.float(), ref.float())
            dk, ik = dists.topk(self.k, dim=-1, largest=False)
            return dk, ik
        r = ref.transpose(1,2).contiguous(); q = query.transpose(1,2).contiguous()
        dists = torch.cdist(q.float(), r.float())
        dk, ik = dists.topk(self.k, dim=-1, largest=False)
        return dk.transpose(1,2), ik.transpose(1,2)
_force('knn_cuda', {'KNN': _KNN})
print('[OK] Mock knn_cuda (forzado)')

_Gridding = _dummy_cls('Gridding'); _GriddingReverse = _dummy_cls('GriddingReverse')
_CubicFS  = _dummy_cls('CubicFeatureSampling'); _GriddingLoss = _dummy_cls('GriddingLoss')
class _EmdModule(nn.Module):
    def forward(self, xyz1, xyz2):
        return (torch.zeros(xyz1.shape[0], device=xyz1.device),
                torch.zeros(xyz1.shape[0], dtype=torch.int32, device=xyz1.device))
for _base, _attrs in [
    ('gridding',               {'Gridding': _Gridding, 'GriddingReverse': _GriddingReverse}),
    ('gridding_loss',          {'GriddingLoss': _GriddingLoss}),
    ('cubic_feature_sampling', {'CubicFeatureSampling': _CubicFS}),
    ('emd',                    {'emd_module': _EmdModule, 'EarthMoverDistance': _EmdModule}),
]:
    for _prefix in ['', 'extensions.']: _inject(_prefix + _base, _attrs)
print('[OK] Mocks gridding / emd / cubic_feature_sampling')
print()

# ── Chamfer para training loop ────────────────────────────────
from extensions.chamfer_dist import ChamferDistanceL1 as _CDL1
_cd_fn = _CDL1()
def chamfer_distance(pred, gt):
    return _cd_fn(pred.contiguous(), gt.contiguous())
print('[OK] Chamfer para training loop')

def subsample_gt(gt, n):
    return gt[:, torch.randperm(gt.size(1), device=gt.device)[:n], :]

# ── Config del modelo ─────────────────────────────────────────
import yaml

cfg_path = '/content/PoinTr/cfgs/PCN_models/PoinTr.yaml'
if not os.path.exists(cfg_path):
    candidates = _glob.glob('/content/PoinTr/cfgs/**/PoinTr.yaml', recursive=True)
    cfg_path = candidates[0] if candidates else None
if cfg_path and os.path.exists(cfg_path):
    with open(cfg_path) as f: raw_cfg = yaml.safe_load(f)
    model_cfg = EasyDict(raw_cfg.get('model', raw_cfg))
    print(f'Config base: {cfg_path}')
else:
    model_cfg = EasyDict({'NAME': 'PoinTr'})
model_cfg.num_pred = 2048; model_cfg.num_query = 128
print(f'Config: num_pred={model_cfg.num_pred}, num_query={model_cfg.num_query}')

# ── Cargar modelo ─────────────────────────────────────────────
try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(model_cfg)
    print('[OK] Modelo cargado via build_model_from_cfg')
except Exception as e1:
    try:
        from models.PoinTr import PoinTr
        model = PoinTr(model_cfg)
        print('[OK] Modelo cargado via PoinTr directa')
    except Exception as e2:
        raise RuntimeError(f'No se pudo inicializar PoinTr.\nError 1: {e1}\nError 2: {e2}')

model = model.to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametros entrenables: {n_params:,} | Dispositivo: {device}')

if device.type == 'cpu':
    raise SystemExit('Sin GPU — necesitas T4/A100.')

# ── Dataset con blacklist (filtrado inline) + CENTRAR_EN_ROTO=False ─────
from E3.dataset import construir_pares, ShapeCompletionDataset

import E3.dataset as _ds
_ds.CENTRAR_EN_ROTO = False
print('[OK] CENTRAR_EN_ROTO=False — roto y completo en el frame original')

_carpetas = ['Datos/sintetico/roturas_centradas', 'Datos/fantastic_breaks/procesado']
_todos = construir_pares(_carpetas)
n_antes = len(_todos)
_todos = [(r, c) for r, c in _todos
          if r.stem.replace('_roto', '') not in blacklist_set]
print(f'Blacklist: {n_antes - len(_todos)} excluidos → {len(_todos)} pares limpios')

_rng = random.Random(42); _rng.shuffle(_todos); _n = len(_todos)
_n_train = int(0.8 * _n); _n_val = int(0.1 * _n)
_pares_train = _todos[:_n_train]
_pares_val   = _todos[_n_train:_n_train + _n_val]
_pares_test  = _todos[_n_train + _n_val:]
print(f'  Train: {len(_pares_train)} | Val: {len(_pares_val)} | Test: {len(_pares_test)}')

train_loader = DataLoader(ShapeCompletionDataset(_pares_train, augmentar=True),
                          batch_size=32, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(ShapeCompletionDataset(_pares_val,   augmentar=False),
                          batch_size=32, shuffle=False, pin_memory=True)
test_loader  = DataLoader(ShapeCompletionDataset(_pares_test,  augmentar=False),
                          batch_size=32, shuffle=False, pin_memory=True)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

# ══════════════════════════════════════════════════════════════
# 3. HIPERPARÁMETROS Y CHECKPOINT RESUME
# ══════════════════════════════════════════════════════════════
EPOCHS   = 200; LR = 1e-4; W_COARSE = 0.5
CKPT_DIR = Path('E3/checkpoints_pointr_v3'); CKPT_DIR.mkdir(parents=True, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
mejor_val = math.inf; train_losses = []; val_losses = []; START_EPOCH = 1

def _find_latest_ckpt():
    drive_ckpts = sorted(Path(RUTA_SALIDA_MODELO).glob('epoch_*.pt')) \
                  if Path(RUTA_SALIDA_MODELO).exists() else []
    local_ckpts = sorted(CKPT_DIR.glob('epoch_*.pt'))
    all_ckpts = drive_ckpts + local_ckpts
    if not all_ckpts: return None
    def _ep(p):
        try: return int(p.stem.split('_')[1])
        except: return 0
    return max(all_ckpts, key=_ep)

latest = _find_latest_ckpt()
if latest:
    print(f'\nReanudando desde checkpoint: {latest}')
    ckpt_data = torch.load(latest, map_location=device, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'])
    optimizer.load_state_dict(ckpt_data['optimizer_state_dict'])
    train_losses = ckpt_data.get('train_losses', [])
    val_losses   = ckpt_data.get('val_losses', [])
    START_EPOCH  = ckpt_data['epoch'] + 1
    mejor_val    = min(val_losses) if val_losses else math.inf
    for _ in range(START_EPOCH - 1): scheduler.step()
    print(f'  Reanudando desde epoca {START_EPOCH}/{EPOCHS} | mejor val={mejor_val:.6f}')
else:
    print('\nEntrenamiento desde epoca 1 (sin checkpoint previo)')

# ── Si el entrenamiento ya esta completo: copiar best.pt de Drive a local ──
if START_EPOCH > EPOCHS:
    print(f'\nEntrenamiento ya completo ({EPOCHS} epocas).')
    drive_best = Path(RUTA_SALIDA_MODELO) / 'best.pt'
    local_best = CKPT_DIR / 'best.pt'
    if not local_best.exists() and drive_best.exists():
        shutil.copy2(drive_best, local_best)
        print(f'[OK] best.pt copiado de Drive a {local_best}')
    elif local_best.exists():
        print(f'[OK] best.pt ya disponible en {local_best}')
    else:
        print(f'[WARN] best.pt no encontrado — revisa Drive: {drive_best}')
    print('Pasa directamente a la Celda 9 para evaluar.')
    raise SystemExit('Entrenamiento ya completado.')

def guardar_ckpt(nombre, epoch):
    data = {'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses, 'val_losses': val_losses,
            'model_cfg': dict(model_cfg)}
    local_path = CKPT_DIR / nombre
    torch.save(data, local_path)
    try:
        Path(RUTA_SALIDA_MODELO).mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_path, Path(RUTA_SALIDA_MODELO) / nombre)
    except Exception as e:
        print(f'  [WARN] No se pudo guardar en Drive: {e}')

print(f'\n{"Epoca":>7}  {"Train":>10}  {"Val":>10}  {"LR":>9}  {"Tiempo":>7}')
print('-' * 52)

for epoch in range(START_EPOCH, EPOCHS + 1):
    t0 = time.time()
    model.train(); total_train = 0.0
    for roto, completo in train_loader:
        roto, completo = roto.to(device), completo.to(device)
        optimizer.zero_grad()
        out    = model(roto)
        coarse = out[0] if isinstance(out, (list, tuple)) else out
        fine   = out[-1] if isinstance(out, (list, tuple)) else out
        gt_c   = subsample_gt(completo, coarse.size(1))
        loss   = chamfer_distance(fine, completo) + W_COARSE * chamfer_distance(coarse, gt_c)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step(); total_train += loss.item()

    model.eval(); total_val = 0.0
    with torch.no_grad():
        for roto, completo in val_loader:
            roto, completo = roto.to(device), completo.to(device)
            out    = model(roto)
            coarse = out[0] if isinstance(out, (list, tuple)) else out
            fine   = out[-1] if isinstance(out, (list, tuple)) else out
            gt_c   = subsample_gt(completo, coarse.size(1))
            total_val += (chamfer_distance(fine, completo) +
                          W_COARSE * chamfer_distance(coarse, gt_c)).item()

    loss_train = total_train / len(train_loader)
    loss_val   = total_val   / len(val_loader)
    elapsed    = time.time() - t0
    train_losses.append(loss_train); val_losses.append(loss_val)
    scheduler.step(); lr_now = scheduler.get_last_lr()[0]

    print(f'{epoch:>4}/{EPOCHS}  {loss_train:>10.6f}  {loss_val:>10.6f}'
          f'  {lr_now:>9.2e}  {elapsed:>6.1f}s')

    if epoch % 10 == 0:
        guardar_ckpt(f'epoch_{epoch:03d}.pt', epoch)

    if loss_val < mejor_val:
        mejor_val = loss_val
        guardar_ckpt('best.pt', epoch)
        print(f'  -> best.pt guardado (val={mejor_val:.6f})')

print(f'\nFin. Mejor val loss: {mejor_val:.6f}')
print(f'Checkpoints en: {CKPT_DIR} y en Drive: {RUTA_SALIDA_MODELO}')

In [ ]:
# ── CELDA 8: Guardar modelo PoinTr v3 en Drive ─────────────────
import shutil, os
from pathlib import Path

os.chdir('/content/TFM')

# Rutas (ya definidas en Celda 3; se redefinen aquí por si se ejecuta directamente)
DRIVE = '/content/drive/MyDrive'
RUTA_SALIDA_MODELO = f'{DRIVE}/Datos_E2_E3/E3/Raquel/modelos/v3_pointr'

Path(RUTA_SALIDA_MODELO).mkdir(parents=True, exist_ok=True)

shutil.copy2('E3/checkpoints_pointr_v3/best.pt', f'{RUTA_SALIDA_MODELO}/best.pt')
print(f'Modelo PoinTr v3 guardado: {RUTA_SALIDA_MODELO}/best.pt')

for ckpt in sorted(Path('E3/checkpoints_pointr_v3').glob('epoch_*.pt')):
    shutil.copy2(ckpt, Path(RUTA_SALIDA_MODELO) / ckpt.name)
    print(f'  + {ckpt.name}')

In [ ]:
# ── CELDA 9: Evaluar PoinTr v3 ─────────────────────────────────
import torch, numpy as np, random
from pathlib import Path
from torch.utils.data import DataLoader
import sys, os

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

# Rutas (ya definidas en Celda 3; se redefinen aquí por si se ejecuta directamente)
DRIVE = '/content/drive/MyDrive'
RUTA_SALIDA_RESULTADOS = f'{DRIVE}/Datos_E2_E3/E3/Raquel/resultados/v3_pointr'
RUTA_BLACKLIST = f'{DRIVE}/Datos_E2_E3/General/blacklist_sintetico_roturas_centradas.txt'

# Recargar blacklist si no está en scope (ejecución directa de esta celda)
if 'blacklist_set' not in vars() or not blacklist_set:
    blacklist_set = set()
    bl = Path(RUTA_BLACKLIST)
    if bl.exists():
        blacklist_set = {l.strip() for l in bl.read_text().splitlines() if l.strip()}
        print(f'Blacklist recargada: {len(blacklist_set)} entradas')
    else:
        print('[WARN] Blacklist no encontrada — evaluando sin filtro')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt_path = 'E3/checkpoints_pointr_v3/best.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'Cargando best.pt — entrenado hasta epoca {ckpt["epoch"]}')

from easydict import EasyDict
try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(EasyDict(ckpt['model_cfg']))
except Exception:
    from models.PoinTr import PoinTr
    model = PoinTr(EasyDict(ckpt['model_cfg']))
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device); model.eval()

from E3.dataset import construir_pares, ShapeCompletionDataset
import E3.dataset as _ds
_ds.CENTRAR_EN_ROTO = False
print('[OK] CENTRAR_EN_ROTO=False para evaluación')

_todos = construir_pares(['Datos/sintetico/roturas_centradas', 'Datos/fantastic_breaks/procesado'])
_todos = [(r, c) for r, c in _todos if r.stem.replace('_roto', '') not in blacklist_set]
_rng = random.Random(42); _rng.shuffle(_todos); _n = len(_todos)
_n_train = int(0.8 * _n); _n_val = int(0.1 * _n)
_pares_test = _todos[_n_train + _n_val:]
test_loader = DataLoader(ShapeCompletionDataset(_pares_test, augmentar=False),
                         batch_size=32, shuffle=False, pin_memory=True)
print(f'Test: {len(_pares_test)} pares')

try:
    from extensions.chamfer_dist import ChamferDistanceL1
    _cf = ChamferDistanceL1()
    def chamfer_eval(pred, gt): return _cf(pred.contiguous(), gt.contiguous()).item()
except Exception:
    def chamfer_eval(pred, gt):
        dist = torch.cdist(pred, gt, p=2)
        return ((dist.min(dim=2).values.mean() + dist.min(dim=1).values.mean()) / 2).item()

def fscore(pred, gt, umbral=0.01):
    dist_pg = torch.cdist(pred, gt, p=2)
    dist_gp = torch.cdist(gt, pred, p=2)
    prec = (dist_pg.min(dim=2).values < umbral).float().mean()
    rec  = (dist_gp.min(dim=2).values < umbral).float().mean()
    if prec + rec < 1e-8: return 0.0
    return (2 * prec * rec / (prec + rec)).item()

cds, fscores = [], []
with torch.no_grad():
    for roto, completo in test_loader:
        roto, completo = roto.to(device), completo.to(device)
        out = model(roto)
        fine = out[-1] if isinstance(out, (list, tuple)) else out
        for i in range(len(roto)):
            p = fine[i:i+1]; g = completo[i:i+1]
            cds.append(chamfer_eval(p, g))
            fscores.append(fscore(p, g))

cd_mean = np.mean(cds); fs_mean = np.mean(fscores)
print(f'\n=== PoinTr v3 — Resultados test ===')
print(f'  CD-L1   : {cd_mean:.4f}')
print(f'  F-Score : {fs_mean:.4f}')
print(f'  Muestras: {len(cds)}')

import shutil
resumen_dir = Path('E3/resultados_pointr_v3'); resumen_dir.mkdir(parents=True, exist_ok=True)
with open(resumen_dir / 'resumen.txt', 'w') as f:
    f.write(f'Modelo: PoinTr v3\nEpoca best: {ckpt["epoch"]}\n')
    f.write(f'CD-L1: {cd_mean:.6f}\nF-Score: {fs_mean:.6f}\nMuestras test: {len(cds)}\n')
Path(RUTA_SALIDA_RESULTADOS).mkdir(parents=True, exist_ok=True)
shutil.copytree('E3/resultados_pointr_v3', RUTA_SALIDA_RESULTADOS, dirs_exist_ok=True)
print(f'Resultados guardados: {RUTA_SALIDA_RESULTADOS}')

In [ ]:
# ── CELDA 10: Comparar todos los modelos E3 ────────────────────
from pathlib import Path

# Rutas (ya definidas en Celda 3; se redefinen aquí por si se ejecuta directamente)
DRIVE   = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'

comparaciones = [
    ('PCN v5 (ref)',   f'{BASE_E3}/resultados/v5_pcn/resumen.txt',    '0.062954', '0.025700'),
    ('PoinTr v2',      f'{BASE_E3}/resultados/v2_pointr/resumen.txt', '0.053300', '0.327800'),
    ('PoinTr v3',      'E3/resultados_pointr_v3/resumen.txt',          None,       None),
]

print('=' * 65)
print(f'{"Modelo":<22}  {"CD-L1":>10}  {"F-Score":>10}  {"vs PCN v5"}')
print('=' * 65)

ref_cd = 0.062954

for nombre, ruta, cd_fallback, fs_fallback in comparaciones:
    if cd_fallback is not None:
        cd_str, fs_str = cd_fallback, fs_fallback
        cd_val = float(cd_fallback)
    elif ruta and Path(ruta).exists():
        txt = Path(ruta).read_text()
        cd_str = fs_str = '?'
        cd_val = None
        for line in txt.splitlines():
            if line.startswith('CD-L1:'):
                cd_str = line.split(':')[1].strip(); cd_val = float(cd_str)
            if line.startswith('F-Score:'):
                fs_str = line.split(':')[1].strip()
    else:
        cd_str = fs_str = 'pendiente'; cd_val = None

    mejora = ''
    if cd_val is not None and nombre != 'PCN v5 (ref)':
        pct = (ref_cd - cd_val) / ref_cd * 100
        mejora = f'{pct:+.1f}%' if abs(pct) > 0.05 else '='

    print(f'{nombre:<22}  {cd_str:>10}  {fs_str:>10}  {mejora}')

print('=' * 65)
print('\nCD-L1 menor = mejor | F-Score mayor = mejor')

In [ ]:
# ── CELDA 11: Curvas de aprendizaje ────────────────────────────
import matplotlib.pyplot as plt
import torch

ckpt = torch.load('E3/checkpoints_pointr_v3/best.pt', map_location='cpu', weights_only=False)
train_losses = ckpt['train_losses']
val_losses   = ckpt['val_losses']
best_epoch   = ckpt['epoch']

fig, ax = plt.subplots(figsize=(10, 4))
epochs = range(1, len(train_losses) + 1)
ax.plot(epochs, train_losses, label='Train', color='#1565C0', alpha=0.8)
ax.plot(epochs, val_losses,   label='Val',   color='#C62828', alpha=0.8)
ax.axvline(best_epoch, color='#2E7D32', linestyle='--', alpha=0.7, label=f'best (e{best_epoch})')
ax.set_xlabel('Epoca')
ax.set_ylabel('Loss (CD-L1 + 0.5*CD-coarse)')
ax.set_title('PoinTr v3 — curvas de aprendizaje (CENTRAR_EN_ROTO=False, 200 épocas)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('E3/resultados_pointr_v3/curvas.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Mejor epoch: {best_epoch} | val loss: {min(val_losses):.6f}')

In [ ]:
# ── CELDA 12: Visualizacion 3D interactiva ─────────────────────
import subprocess
subprocess.run(['pip', 'install', 'plotly', '--quiet'])

import plotly.graph_objects as go
import numpy as np, torch, random, sys, os
from pathlib import Path
from torch.utils.data import DataLoader

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

from easydict import EasyDict
from E3.dataset import construir_pares, ShapeCompletionDataset
import E3.dataset as _ds
_ds.CENTRAR_EN_ROTO = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt = torch.load('E3/checkpoints_pointr_v3/best.pt', map_location=device, weights_only=False)
try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(EasyDict(ckpt['model_cfg']))
except Exception:
    from models.PoinTr import PoinTr
    model = PoinTr(EasyDict(ckpt['model_cfg']))
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device); model.eval()
print(f'Modelo cargado — epoca {ckpt["epoch"]}')

_todos = construir_pares(['Datos/sintetico/roturas_centradas', 'Datos/fantastic_breaks/procesado'])
_todos = [(r, c) for r, c in _todos if r.stem.replace('_roto', '') not in blacklist_set]
_rng = random.Random(42); _rng.shuffle(_todos); _n = len(_todos)
_n_train = int(0.8 * _n); _n_val = int(0.1 * _n)
_pares_test = _todos[_n_train + _n_val:]
test_loader = DataLoader(ShapeCompletionDataset(_pares_test, augmentar=False),
                         batch_size=32, shuffle=False, pin_memory=True)

def chamfer_np(a, b):
    a, b = torch.tensor(a).unsqueeze(0), torch.tensor(b).unsqueeze(0)
    dist = torch.cdist(a, b, p=2)
    return ((dist.min(2).values.mean() + dist.min(1).values.mean()) / 2).item()

rotos, gts, preds, cds = [], [], [], []
with torch.no_grad():
    for roto_b, gt_b in test_loader:
        out = model(roto_b.to(device))
        pred_b = (out[-1] if isinstance(out, (list, tuple)) else out).cpu()
        for i in range(len(roto_b)):
            rotos.append(roto_b[i].numpy())
            gts.append(gt_b[i].numpy())
            preds.append(pred_b[i].numpy())
            cds.append(chamfer_np(pred_b[i].numpy(), gt_b[i].numpy()))

cds_arr = np.array(cds)
orden   = np.argsort(cds_arr)
indices = list(orden[:3]) + list(orden[-3:])
titulos = ['Mejor 1', 'Mejor 2', 'Mejor 3', 'Peor 1', 'Peor 2', 'Peor 3']
print(f'CD — mejor: {cds_arr[orden[0]]:.4f} | peor: {cds_arr[orden[-1]]:.4f} | media: {cds_arr.mean():.4f}')

for idx, titulo in zip(indices, titulos):
    fig = go.Figure([
        go.Scatter3d(x=rotos[idx][:,0], y=rotos[idx][:,1], z=rotos[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#4C72B0', opacity=0.55), name='Rota'),
        go.Scatter3d(x=gts[idx][:,0],   y=gts[idx][:,1],   z=gts[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#55A868', opacity=0.35), name='GT'),
        go.Scatter3d(x=preds[idx][:,0], y=preds[idx][:,1], z=preds[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#C44E52', opacity=0.85), name='PoinTr'),
    ])
    fig.update_layout(
        title=f'{titulo} — CD={cds[idx]:.4f}',
        scene=dict(xaxis=dict(range=[-1,1], showticklabels=False),
                   yaxis=dict(range=[-1,1], showticklabels=False),
                   zaxis=dict(range=[-1,1], showticklabels=False),
                   aspectmode='cube'),
        height=500, margin=dict(l=0,r=0,b=0,t=40)
    )
    fig.show()